# 04. Atribuir 1 CPF por Censo

Lista operacional: cada Censo recebe **no máximo um** CPF. Independente do
[`03_avaliar.ipynb`](03_avaliar.ipynb) — só precisa do `02b_aplicar` (predictions +
clusters) e da coorte.

Candidatos: par com `match_probability ≥ THRESHOLD_AVALIACAO`, Censo e CPF no
**mesmo** `cluster_id`, tipo do cluster `<> 'outros'`. Mega-cluster N×M e 1 Censo
× N CPFs (ambos caem em `outros`) ficam de fora.

**Greedy 1-1:** ordena os pares por score (empate: `unique_id_censo`) e atribui se
Censo e CPF ainda estão livres. Em `1_cpf_n_censo` fica o Censo de maior score.
Não é “melhor CPF por Censo e depois drop de colisão” (C1–X 0,99 / C2–X 0,98 /
C2–Y 0,97 deixaria C2 sem par). Um 2×2 real é `outros` e não entra na lista.

Sem atribuição: sem par ≥ T, só em `outros`, ou perdeu o greedy.

Saída: `splink_atribuicao.parquet`. Contra a ouro 1:1 do subset:
`recall_atribuicao`, CPF errado, e sem atribuição por motivo.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    METRICAS_ATRIBUICAO,
    SPLINK_ATRIBUICAO,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    export_parquet,
    get_connection,
    materialize_atribuicao,
    materialize_cluster_composicao,
    materialize_gt_no_subset,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

def display_metricas(df_uma_linha):
    """Uma linha de métricas: inteiros com milhar, o resto com 4 casas (evita 1.43e+06)."""
    row = df_uma_linha.iloc[0]
    linhas = []
    for k, v in row.items():
        if v is None or (isinstance(v, float) and pd.isna(v)):
            txt = ''
        elif isinstance(v, bool):
            txt = v
        elif pd.api.types.is_number(v) and not isinstance(v, bool):
            fv = float(v)
            if abs(fv - round(fv)) < 1e-12 and abs(fv) >= 1:
                txt = f'{int(round(fv)):,}'
            else:
                txt = f'{fv:.4f}'
        else:
            txt = v
        linhas.append({'metrica': k, 'valor': txt})
    display(pd.DataFrame(linhas))


print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
require_input(SPLINK_CLUSTERS, label='SPLINK_CLUSTERS (rode o 02b_aplicar antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
print('Threshold:', THRESHOLD_AVALIACAO)


## 1. Predictions, clusters, ouro

`unique_id_l` = Censo, `unique_id_r` = CPF (o parquet do Splink pode vir invertido).
`cluster_composicao` é a mesma classificação do 03 (`singleton`, `1_para_1`,
`1_cpf_n_censo`, `outros`).


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_l,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_r,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE splink_clusters AS
SELECT * FROM read_parquet('{SPLINK_CLUSTERS}')
''')

materialize_cluster_composicao(con)
display(con.execute('''
SELECT tipo, COUNT(*) AS n_clusters, SUM(n_censo) AS n_censo, SUM(n_cpf) AS n_cpf
FROM cluster_composicao
GROUP BY tipo
ORDER BY n_clusters DESC
''').df())

counts = materialize_gt_no_subset(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_gt = counts['n_gt_no_subset']
print('Ouro 1:1 no subset:', f'{n_gt:,}')
if n_gt == 0:
    raise RuntimeError(
        'Nenhum par da coorte caiu no subset — confira o filtro geográfico do NB00.'
    )


## 2. Greedy 1 CPF por Censo

`materialize_atribuicao` filtra pares ≥ corte no mesmo cluster `<> outros` e
resolve conflitos no pandas (score descendente).


In [ ]:
stats = materialize_atribuicao(con, threshold=THRESHOLD_AVALIACAO)
print(
    f"Candidatos ≥ {stats['threshold']}: {stats['n_candidatos']:,} | "
    f"atribuidos: {stats['n_atribuidos']:,}"
)
export_parquet(con, 'atribuicao_censo_cpf', path=SPLINK_ATRIBUICAO)
print('Parquet:', SPLINK_ATRIBUICAO)

display(con.execute('''
SELECT tipo_cluster, COUNT(*) AS n
FROM atribuicao_censo_cpf
GROUP BY 1
ORDER BY n DESC
''').df())
display(con.execute('''
SELECT unique_id_censo, unique_id_cpf, match_probability, cluster_id, tipo_cluster
FROM atribuicao_censo_cpf
ORDER BY match_probability DESC
LIMIT 20
''').df())


Cobertura no subset (todos os Censos, não só a ouro). Motivo de quem ficou de fora:
`sem_par` (nenhum CPF ≥ T no parquet), `mega_cluster` (tipo `outros`), `greedy`
(tinha candidato válido e perdeu o CPF ou o Censo já estava tomado).


In [ ]:
motivo_expr = f'''
CASE
    WHEN a.unique_id_cpf IS NOT NULL THEN 'atribuido'
    WHEN COALESCE(comp.tipo, 'sem_cluster') = 'outros' THEN 'mega_cluster'
    WHEN COALESCE(melhor.pmax, 0) < {THRESHOLD_AVALIACAO} THEN 'sem_par'
    ELSE 'greedy'
END
'''

con.execute(f'''
CREATE OR REPLACE TABLE censo_motivo AS
WITH censo AS (
    SELECT unique_id AS unique_id_censo
    FROM {SPLINK_INPUT_VIEW}
    WHERE origem = 'censo'
),
melhor AS (
    SELECT unique_id_l, MAX(match_probability) AS pmax
    FROM splink_predictions
    GROUP BY 1
)
SELECT
    c.unique_id_censo,
    a.unique_id_cpf,
    a.match_probability,
    sc.cluster_id,
    comp.tipo AS tipo_cluster,
    {motivo_expr} AS motivo
FROM censo c
LEFT JOIN atribuicao_censo_cpf a ON a.unique_id_censo = c.unique_id_censo
LEFT JOIN splink_clusters sc ON sc.unique_id = c.unique_id_censo
LEFT JOIN cluster_composicao comp ON comp.cluster_id = sc.cluster_id
LEFT JOIN melhor ON melhor.unique_id_l = c.unique_id_censo
''')

n_censo = con.execute('SELECT COUNT(*) FROM censo_motivo').fetchone()[0]
n_ok = con.execute(
    "SELECT COUNT(*) FROM censo_motivo WHERE motivo = 'atribuido'"
).fetchone()[0]
print(
    f'Dos {n_censo:,} Censos no subset, {n_ok:,} '
    f'({100.0 * n_ok / n_censo if n_censo else 0:.2f}%) receberam CPF.'
)
display(con.execute('''
SELECT motivo, COUNT(*) AS n
FROM censo_motivo
GROUP BY 1
ORDER BY n DESC
''').df())


## 3. Versus ouro 1:1

`recall_atribuicao` = fração da ouro cujo CPF atribuído **é** o X.
`n_ouro_cpf_errado` = atribuiu outro CPF. Sem atribuição repartido pelos mesmos
motivos da cobertura.

Amostra de CPF errado: Censo, CPF ouro e CPF atribuído (nome, sexo, data, idade)
e, se o par ouro estiver no parquet, o score dele ao lado do score do atribuído.


In [ ]:
vs = con.execute(f'''
SELECT
    CAST(COUNT(*) AS BIGINT) AS n_ouro,
    CAST(SUM(CASE WHEN a.unique_id_cpf IS NOT NULL THEN 1 ELSE 0 END) AS BIGINT)
        AS n_ouro_com_atribuicao,
    CAST(SUM(CASE WHEN a.unique_id_cpf = gt.unique_id_cpf THEN 1 ELSE 0 END) AS BIGINT)
        AS n_ouro_cpf_certo,
    ROUND(
        1.0 * SUM(CASE WHEN a.unique_id_cpf = gt.unique_id_cpf THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        4
    ) AS recall_atribuicao,
    CAST(SUM(CASE
        WHEN a.unique_id_cpf IS NOT NULL
         AND a.unique_id_cpf <> gt.unique_id_cpf THEN 1
        ELSE 0
    END) AS BIGINT) AS n_ouro_cpf_errado,
    CAST(SUM(CASE WHEN a.unique_id_cpf IS NULL THEN 1 ELSE 0 END) AS BIGINT)
        AS n_ouro_sem_atribuicao
FROM gt_no_subset gt
LEFT JOIN atribuicao_censo_cpf a ON a.unique_id_censo = gt.unique_id_censo
''').df()
vs['threshold_avaliacao'] = THRESHOLD_AVALIACAO
display_metricas(vs)

motivos_ouro = con.execute('''
SELECT m.motivo, CAST(COUNT(*) AS BIGINT) AS n
FROM gt_no_subset gt
JOIN censo_motivo m ON m.unique_id_censo = gt.unique_id_censo
WHERE m.motivo <> 'atribuido'
GROUP BY 1
ORDER BY n DESC
''').df()
print('Ouro sem atribuição por motivo:')
display(motivos_ouro)

print('Ouro com CPF errado (amostra, três lados):')
display(con.execute(f'''
SELECT
    gt.unique_id_censo,
    gt.unique_id_cpf AS unique_id_ouro,
    a.unique_id_cpf AS unique_id_atribuido,
    a.match_probability AS p_atribuido,
    po.match_probability AS p_ouro,
    a.tipo_cluster,
    ca.nome_completo AS nome_censo,
    ouro.nome_completo AS nome_ouro,
    atr.nome_completo AS nome_atribuido,
    ca.sexo AS sexo_censo,
    ouro.sexo AS sexo_ouro,
    atr.sexo AS sexo_atribuido,
    ca.data_nascimento AS dob_censo,
    ouro.data_nascimento AS dob_ouro,
    atr.data_nascimento AS dob_atribuido,
    ca.idade AS idade_censo,
    ouro.idade AS idade_ouro,
    atr.idade AS idade_atribuido
FROM gt_no_subset gt
JOIN atribuicao_censo_cpf a ON a.unique_id_censo = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = gt.unique_id_cpf
JOIN {SPLINK_INPUT_VIEW} atr ON atr.unique_id = a.unique_id_cpf
LEFT JOIN splink_predictions po
  ON po.unique_id_l = gt.unique_id_censo AND po.unique_id_r = gt.unique_id_cpf
WHERE a.unique_id_cpf <> gt.unique_id_cpf
ORDER BY a.match_probability DESC
LIMIT 30
''').df())


## 4. Atribuições 1_para_1 com nome ou data diferentes

Só `tipo_cluster = '1_para_1'`. Discordância com `IS DISTINCT FROM` (NULL conta
como diferente): `nome_completo` **ou** `data_nascimento` do Censo vs CPF
atribuído. Quebra: só nome / só DOB / ambos. Amostra traz também `nome_completo_phon`.


In [ ]:
metricas_1a1 = con.execute(f'''
SELECT
    CAST(COUNT(*) AS BIGINT) AS n_1_para_1,
    CAST(COALESCE(SUM(CASE
        WHEN ca.nome_completo IS DISTINCT FROM pb.nome_completo
          OR ca.data_nascimento IS DISTINCT FROM pb.data_nascimento
        THEN 1 ELSE 0
    END), 0) AS BIGINT) AS n_nome_ou_dob_diff,
    ROUND(
        100.0 * SUM(CASE
            WHEN ca.nome_completo IS DISTINCT FROM pb.nome_completo
              OR ca.data_nascimento IS DISTINCT FROM pb.data_nascimento
            THEN 1 ELSE 0
        END) / NULLIF(COUNT(*), 0),
        2
    ) AS pct_nome_ou_dob_diff,
    CAST(COALESCE(SUM(CASE
        WHEN ca.nome_completo IS DISTINCT FROM pb.nome_completo
         AND ca.data_nascimento IS NOT DISTINCT FROM pb.data_nascimento
        THEN 1 ELSE 0
    END), 0) AS BIGINT) AS n_so_nome,
    CAST(COALESCE(SUM(CASE
        WHEN ca.nome_completo IS NOT DISTINCT FROM pb.nome_completo
         AND ca.data_nascimento IS DISTINCT FROM pb.data_nascimento
        THEN 1 ELSE 0
    END), 0) AS BIGINT) AS n_so_dob,
    CAST(COALESCE(SUM(CASE
        WHEN ca.nome_completo IS DISTINCT FROM pb.nome_completo
         AND ca.data_nascimento IS DISTINCT FROM pb.data_nascimento
        THEN 1 ELSE 0
    END), 0) AS BIGINT) AS n_ambos
FROM atribuicao_censo_cpf a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
WHERE a.tipo_cluster = '1_para_1'
''').df()
display_metricas(metricas_1a1)

print('Amostra 1_para_1 com nome ou data diferentes:')
display(con.execute(f'''
SELECT
    a.unique_id_censo,
    a.unique_id_cpf,
    a.match_probability,
    (ca.nome_completo IS DISTINCT FROM pb.nome_completo) AS nome_diff,
    (ca.data_nascimento IS DISTINCT FROM pb.data_nascimento) AS dob_diff,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.nome_completo_phon AS nome_phon_censo,
    pb.nome_completo_phon AS nome_phon_cpf,
    ca.sexo AS sexo_censo,
    pb.sexo AS sexo_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf
FROM atribuicao_censo_cpf a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
WHERE a.tipo_cluster = '1_para_1'
  AND (
    ca.nome_completo IS DISTINCT FROM pb.nome_completo
    OR ca.data_nascimento IS DISTINCT FROM pb.data_nascimento
  )
ORDER BY a.match_probability DESC
LIMIT 30
''').df())


In [ ]:
metricas = vs.copy()
metricas['n_censo_subset'] = n_censo
metricas['n_censo_atribuido'] = n_ok
metricas['n_candidatos'] = stats['n_candidatos']
metricas['n_atribuidos'] = stats['n_atribuidos']
metricas['n_nao_1a1_descartada'] = counts['n_nao_1a1_descartada']
for motivo in ('sem_par', 'mega_cluster', 'greedy'):
    if len(motivos_ouro) and 'motivo' in motivos_ouro.columns:
        n = int(motivos_ouro.loc[motivos_ouro['motivo'] == motivo, 'n'].sum())
    else:
        n = 0
    metricas[f'n_ouro_{motivo}'] = n
for col in (
    'n_1_para_1',
    'n_nome_ou_dob_diff',
    'pct_nome_ou_dob_diff',
    'n_so_nome',
    'n_so_dob',
    'n_ambos',
):
    metricas[col] = metricas_1a1[col].iloc[0]
metricas.to_csv(METRICAS_ATRIBUICAO, index=False)
print('Métricas:', METRICAS_ATRIBUICAO)
print('Lista:', SPLINK_ATRIBUICAO)
display_metricas(metricas)
con.close()
